In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Generate Synthetic FinTech Data (10,000 users)
np.random.seed(42)
n_users = 10000

# Base features
support_tickets = np.random.poisson(lam=1.2, size=n_users)
days_since_last_login = np.random.poisson(lam=14, size=n_users)
avg_txn_value = np.random.normal(loc=500, scale=150, size=n_users).round(2)
cac = np.random.normal(loc=50, scale=10, size=n_users).round(2)

# Engineered "Velocity" features (The secret to high ROC-AUC)
prev_month_txns = np.random.poisson(lam=25, size=n_users)
current_month_txns = np.random.poisson(lam=20, size=n_users)

# Calculate the drop in engagement (Positive number = drop in usage)
txn_drop_ratio = (prev_month_txns - current_month_txns) / (prev_month_txns + 1)

data = {
    'user_id': range(1, n_users + 1),
    'current_month_txns': current_month_txns,
    'prev_month_txns': prev_month_txns,
    'txn_drop_ratio': txn_drop_ratio.round(3),
    'avg_txn_value': avg_txn_value,
    'support_tickets': support_tickets,
    'days_since_last_login': days_since_last_login,
    'cac': cac
}
df = pd.DataFrame(data)

# Sharpened Non-Linear Signal for Churn using a Sigmoid curve
# High support tickets + massive drop in txns + hasn't logged in recently = High Risk
base_risk = (df['support_tickets'] * 0.9) + (df['txn_drop_ratio'] * 4.5) + (df['days_since_last_login'] * 0.15) - 3.5

# Convert risk score to a realistic probability between 0 and 1
churn_prob = 1 / (1 + np.exp(-base_risk))
df['churn'] = np.random.binomial(1, churn_prob)


print(f"Overall Churn Rate: {df['churn'].mean() * 100:.1f}%")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare Data
X = df.drop(['user_id', 'churn'], axis=1)
y = df['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 1. Train Random Forest (The Baseline) ---
# We use class_weight='balanced' to handle the natural churn imbalance
rf_model = RandomForestClassifier(
    n_estimators=350, 
    max_depth=5, 
    class_weight='balanced', 
    random_state=42
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

# --- 2. Train XGBoost (The Challenger) ---
imbalance_ratio = sum(y_train == 0) / sum(y_train == 1)
xgb_model = xgb.XGBClassifier(
    max_depth=5, 
    learning_rate=0.03, 
    n_estimators=350, 
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=imbalance_ratio,
    eval_metric='logloss', 
    random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

# Add XGBoost probabilities back to the main dataframe for downstream LTV calculations
df['predicted_churn_prob'] = np.clip(xgb_model.predict_proba(X)[:, 1], 0.01, 0.99)

# --- 3. Evaluate and Compare Metrics ---
comparison_data = {
    'Metric': ['Accuracy', 'ROC-AUC', 'Precision (Churn)', 'Recall (Churn)', 'F1-Score'],
    'Random Forest': [
        accuracy_score(y_test, rf_preds),
        roc_auc_score(y_test, rf_probs),
        precision_score(y_test, rf_preds),
        recall_score(y_test, rf_preds),
        f1_score(y_test, rf_preds)
    ],
    'XGBoost': [
        accuracy_score(y_test, xgb_preds),
        roc_auc_score(y_test, xgb_probs),
        precision_score(y_test, xgb_preds),
        recall_score(y_test, xgb_preds),
        f1_score(y_test, xgb_preds)
    ]
}
comparison_df = pd.DataFrame(comparison_data)

print("--- Model Performance Comparison ---")
print(comparison_df.round(3).to_string(index=False))

# --- 4. Plot the Comparison ---
comparison_df_melted = comparison_df.melt(id_vars='Metric', var_name='Model', value_name='Score')

plt.figure(figsize=(10, 5))
sns.barplot(x='Metric', y='Score', hue='Model', data=comparison_df_melted, palette='Set2')
plt.title('Random Forest vs. XGBoost: Churn Prediction Metrics', fontsize=14, pad=15)
plt.ylim(0.6, 1.0) # Zoomed in to clearly see the differences
plt.ylabel('Performance Score', fontsize=12)
plt.xlabel('Evaluation Metric', fontsize=12)
plt.legend(loc='lower right')

# Add numbers on top of the bars for clarity
for p in plt.gca().patches:
    plt.gca().annotate(f"{p.get_height():.3f}", 
                       (p.get_x() + p.get_width() / 2., p.get_height()), 
                       ha='center', va='center', xytext=(0, 5), 
                       textcoords='offset points', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate Revenue Metrics using the correct column name: 'current_month_txns'
df['monthly_revenue'] = (df['current_month_txns'] * df['avg_txn_value']) * 0.02

# Advanced LTV Calculation: Gross Margin / Personalized Churn Probability
gross_margin = 0.80 
df['estimated_ltv'] = (df['monthly_revenue'] * gross_margin) / df['predicted_churn_prob']

# Create LTV to CAC Ratio
df['ltv_cac_ratio'] = df['estimated_ltv'] / df['cac']

# Segment Users based on Profitability Multiples
df['user_segment'] = pd.qcut(df['ltv_cac_ratio'], q=3, labels=['Low Value', 'Medium Value', 'High Value'])

print("\n--- Advanced LTV/CAC Optimization Segments ---")
summary = df.groupby('user_segment', observed=False)[['cac', 'estimated_ltv', 'ltv_cac_ratio']].mean().round(2)
print(summary)

# PLOT 3: Unit Economics (LTV vs CAC) by Segment
# This plot predicts the financial outcome of your segmentation strategy
plt.figure(figsize=(10, 6))
x_labels = summary.index
cac_values = summary['cac']
ltv_values = summary['estimated_ltv']

x = np.arange(len(x_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
rects1 = ax.bar(x - width/2, cac_values, width, label='Customer Acquisition Cost (CAC)', color='lightcoral')
rects2 = ax.bar(x + width/2, ltv_values, width, label='Estimated Lifetime Value (LTV)', color='dodgerblue')

ax.set_ylabel('Dollar Amount ($)', fontsize=12)
ax.set_title('Unit Economics: Average CAC vs. Predicted LTV by Segment', fontsize=14, pad=15)
ax.set_xticks(x)
ax.set_xticklabels(x_labels, fontsize=11)
ax.legend()

# Add value labels on top of the bars
ax.bar_label(rects1, padding=3, fmt='$%.0f')
ax.bar_label(rects2, padding=3, fmt='$%.0f')

plt.tight_layout()
plt.show()